# Answer Quality Filter: Training and Evaluation

**Objective:** Train a DeBERTa-v3-small binary classifier on labeled_asqa.csv
to accept correct answers and reject hallucinated ones, then evaluate against
a no-filter baseline on a held-out test set.

In [1]:
import sys, json, logging
import numpy as np
import pandas as pd
sys.path.append("..")

from src.filtering.data_split import load_and_split
from src.filtering.learned_filter import AnswerQualityClassifier, train_classifier
from src.filtering.filter_evaluator import FilterEvaluator

logging.basicConfig(level=logging.INFO)
SEED = 42

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
RAGAS not installed. Install with: pip install ragas


## 1. Data Split

In [2]:
train_df, val_df, test_df = load_and_split("../data/asqa/labeled_asqa.csv", seed=SEED)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Label distribution (train): {train_df['label'].value_counts().to_dict()}")

INFO:src.filtering.data_split:Loaded 8706 samples from ../data/asqa/labeled_asqa.csv
INFO:src.filtering.data_split:Found 4353 unique base question IDs
INFO:src.filtering.data_split:Train: 5570 samples (pos=2785, neg=2785)
INFO:src.filtering.data_split:Val: 1394 samples (pos=697, neg=697)
INFO:src.filtering.data_split:Test: 1742 samples (pos=871, neg=871)


Train: 5570, Val: 1394, Test: 1742
Label distribution (train): {1: 2785, 0: 2785}


In [ ]:
from src.filtering.learned_filter import _extract_top1_context

for df in [train_df, val_df, test_df]:
    df["context_text"] = df["context"].apply(_extract_top1_context)

print(f"Context extracted. Sample:")
print(f"  Q: {train_df['question'].iloc[0][:80]}")
print(f"  C: {train_df['context_text'].iloc[0][:80]}")
print(f"  A: {train_df['answer'].iloc[0][:80]}")
print(f"  L: {train_df['label'].iloc[0]}")

## 2. Train Classifier

In [ ]:
import shutil
from pathlib import Path

model_dir = Path("../models/answer_filter")
if model_dir.exists():
    shutil.rmtree(model_dir)
    print(f"Removed old model at {model_dir}, retraining...")

model_dir = train_classifier(
    train_df, val_df,
    output_dir=str(model_dir),
)

INFO:src.filtering.learned_filter:Extracting top-1 context for training (dropout=0.15)


Removed old model at ..\models\answer_filter, retraining...


INFO:src.filtering.learned_filter:Context extracted: 5570/5570 train samples have non-empty context
INFO:src.filtering.learned_filter:DIAGNOSTIC [train]: labels={1: 2785, 0: 2785}, NaN(label=0, q=0, a=0), label_dtype=int64
INFO:src.filtering.learned_filter:DIAGNOSTIC [val]: labels={1: 697, 0: 697}, NaN(label=0, q=0, a=0), label_dtype=int64
INFO:src.filtering.learned_filter:Training config: {
  "model_name": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
  "model_path": "models/answer_filter",
  "threshold": 0.5,
  "max_length": 384,
  "batch_size": 16,
  "num_epochs": 10,
  "learning_rate": 2e-05,
  "classifier_lr": 0.001,
  "warmup_ratio": 0.1,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "label_smoothing": 0.0,
  "early_stopping_patience": 3,
  "fp16": false,
  "save_total_limit": 3,
  "seed": 42,
  "use_context": true,
  "context_dropout": 0.15
}
INFO:src.filtering.learned_filter:Model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli  ->  ..\models\answer_filter
INFO:src.filterin

Epoch,Training Loss,Validation Loss


## 3. Threshold Tuning (Validation Set)

In [ ]:
clf = AnswerQualityClassifier(str(model_dir))
evaluator = FilterEvaluator()

val_decisions = clf.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
val_confidences = [d.confidence for d in val_decisions]
val_labels = val_df["label"].tolist()

sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in val_confidences]
    r = evaluator.evaluate(preds, val_labels)
    sweep.append({
        "threshold": round(float(t), 2),
        "f1": r.f1,
        "precision": r.precision,
        "recall": r.recall,
        "accuracy": r.accuracy,
        "rejection_recall": r.rejection_recall,
        "rejection_rate": r.rejection_rate,
    })

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
best_threshold = best_row["threshold"]
print(f"Best threshold: {best_threshold} (F1={best_row['f1']:.4f})")
sweep_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", marker="o")
ax.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall", marker="s")
ax.plot(sweep_df["threshold"], sweep_df["f1"], label="F1", marker="^", linewidth=2)
ax.axvline(best_threshold, color="gray", linestyle="--", label=f"Best t={best_threshold}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Tuning (Validation Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Final Evaluation (Test Set)

In [ ]:
test_decisions = clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
test_preds = [d.confidence >= best_threshold for d in test_decisions]
test_labels = test_df["label"].tolist()

learned_result = evaluator.evaluate(test_preds, test_labels)
baseline_result = evaluator.compute_no_filter_baseline(test_labels)

comparison = evaluator.compare(
    {"No Filter": baseline_result, "Learned Filter": learned_result},
    save_path="../results/filter_comparison.json",
)
pd.DataFrame(comparison).set_index("strategy").round(4)

## 5. Error Analysis

In [ ]:
test_df = test_df.copy()
test_df["predicted"] = test_preds
test_df["confidence"] = [d.confidence for d in test_decisions]

fp = test_df[(test_df["label"] == 0) & (test_df["predicted"] == True)]
fn = test_df[(test_df["label"] == 1) & (test_df["predicted"] == False)]
print(f"False Positives (hallucinations accepted): {len(fp)}")
print(f"False Negatives (correct answers rejected): {len(fn)}")
print(f"\nSample False Positives:")
for _, row in fp.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")
print(f"\nSample False Negatives:")
for _, row in fn.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")

## 6. Apply to Real RAG Outputs

In [ ]:
rag_df = pd.read_csv("../results/asqa_normal_rag_predictions.csv")
if "context" in rag_df.columns:
    rag_df["context_text"] = rag_df["context"].apply(_extract_top1_context)
else:
    rag_df["context_text"] = ""
rag_decisions = clf.predict_batch(rag_df["context_text"].tolist(), rag_df["predicted_answer"].tolist())
rag_df["filter_accept"] = [d.accept for d in rag_decisions]
rag_df["filter_confidence"] = [round(d.confidence, 4) for d in rag_decisions]

n_accept = rag_df["filter_accept"].sum()
print(f"Accepted: {n_accept}/{len(rag_df)} ({100*n_accept/len(rag_df):.1f}%)")
print(f"Rejected: {len(rag_df)-n_accept}/{len(rag_df)} ({100*(len(rag_df)-n_accept)/len(rag_df):.1f}%)")

rag_df.to_csv("../results/rag_predictions_filtered.csv", index=False)

## 7. Ablation: Training Data Size

Train on 25%, 50%, 75%, 100% of the data to measure how much labelled data the filter needs.

In [ ]:
fractions = [0.25, 0.50, 0.75, 1.0]
data_size_rows = []

for frac in fractions:
    tag = f"data_{int(frac * 100)}pct"
    n_samples = int(len(train_df) * frac)
    subset = train_df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag} ({n_samples} samples)...")
        train_classifier(subset, val_df, output_dir=str(out_dir))

    abl_clf = AnswerQualityClassifier(str(out_dir))
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["context_text"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    data_size_rows.append({"fraction": frac, "n_train": n_samples, "threshold": best_abl_t,
                           "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ds_df = pd.DataFrame(data_size_rows)
ds_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ds_df["fraction"] * 100, ds_df["f1"], marker="o", label="F1", linewidth=2)
ax.plot(ds_df["fraction"] * 100, ds_df["accuracy"], marker="s", label="Accuracy")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Score")
ax.set_title("Ablation: Training Data Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Ablation: Max Sequence Length

Compare max_length 256, 384, 512 to measure the trade-off between speed and accuracy.

In [ ]:
max_lengths = [256, 384, 512]
ml_rows = []

for ml in max_lengths:
    tag = f"maxlen_{ml}"
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag}...")
        train_classifier(
            train_df, val_df,
            output_dir=str(out_dir),
            config_overrides={"max_length": ml},
        )

    abl_clf = AnswerQualityClassifier(str(out_dir))
    abl_clf.max_length = ml
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["context_text"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    ml_rows.append({"max_length": ml, "threshold": best_abl_t,
                    "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ml_df = pd.DataFrame(ml_rows)
ml_df

## 9. Ablation: Model Size

Compare DeBERTa-v3-small (44M), base (86M), and large (304M) to measure
the effect of parametric knowledge on closed-book answer verification.

In [ ]:
model_variants = [
    ("microsoft/deberta-v3-small", "deberta_small"),
    ("microsoft/deberta-v3-base", "deberta_base"),
    ("microsoft/deberta-v3-large", "deberta_large"),
]
model_rows = []

for model_id, tag in model_variants:
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag} ({model_id})...")
        train_classifier(
            train_df, val_df,
            output_dir=str(out_dir),
            config_overrides={"model_name": model_id},
        )

    abl_clf = AnswerQualityClassifier(str(out_dir))
    abl_decisions = abl_clf.predict_batch(
        val_df["context_text"].tolist(), val_df["answer"].tolist()
    )
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)

    test_dec = abl_clf.predict_batch(
        test_df["context_text"].tolist(), test_df["answer"].tolist()
    )
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    model_rows.append({
        "model": tag, "threshold": best_abl_t,
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy, "rejection_recall": r.rejection_recall,
        "rejection_rate": r.rejection_rate,
    })
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

model_df = pd.DataFrame(model_rows)
model_df

## 10. NLI Zero-Shot Filter (no training)

Use a pre-trained NLI model as a zero-shot answer quality filter.
Frames the task as: "Does the question entail the answer?"

In [ ]:
from src.filtering.nli_filter import NLIAnswerFilter

nli_filter = NLIAnswerFilter()

nli_val_decisions = nli_filter.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
nli_val_conf = [d.confidence for d in nli_val_decisions]
nli_val_labels = val_df["label"].tolist()

nli_sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in nli_val_conf]
    r = evaluator.evaluate(preds, nli_val_labels)
    nli_sweep.append({
        "threshold": round(float(t), 2),
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy,
        "rejection_recall": r.rejection_recall,
        "rejection_rate": r.rejection_rate,
    })

nli_sweep_df = pd.DataFrame(nli_sweep)
nli_best_row = nli_sweep_df.loc[nli_sweep_df["f1"].idxmax()]
nli_best_threshold = nli_best_row["threshold"]
print(f"NLI Best threshold: {nli_best_threshold} (F1={nli_best_row['f1']:.4f})")

nli_test_decisions = nli_filter.predict_batch(
    test_df["context_text"].tolist(), test_df["answer"].tolist()
)
nli_test_preds = [d.confidence >= nli_best_threshold for d in nli_test_decisions]
nli_result = evaluator.evaluate(nli_test_preds, test_df["label"].tolist())

nli_comparison = evaluator.compare(
    {
        "No Filter": baseline_result,
        "Learned Filter": learned_result,
        "NLI Zero-Shot": nli_result,
    },
    save_path="../results/filter_comparison_with_nli.json",
)
pd.DataFrame(nli_comparison).set_index("strategy").round(4)

## 11. Ensemble Filter (DeBERTa + NLI + Lexical features)

Combine the learned DeBERTa confidence, NLI entailment score, and
lexical features (token overlap, answer length) into a logistic
regression meta-classifier.

In [ ]:
from src.filtering.ensemble_filter import EnsembleFilter

ensemble = EnsembleFilter(
    deberta_clf=clf,
    nli_filter=nli_filter,
)

train_metrics = ensemble.fit(
    train_df["context_text"].tolist(),
    train_df["answer"].tolist(),
    train_df["label"].tolist(),
)
print(f"Ensemble train metrics: {train_metrics}")

ensemble_val_decisions = ensemble.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
ensemble_val_conf = [d.confidence for d in ensemble_val_decisions]

ens_sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in ensemble_val_conf]
    r = evaluator.evaluate(preds, val_df["label"].tolist())
    ens_sweep.append({
        "threshold": round(float(t), 2),
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy,
    })

ens_sweep_df = pd.DataFrame(ens_sweep)
ens_best_row = ens_sweep_df.loc[ens_sweep_df["f1"].idxmax()]
ens_best_threshold = ens_best_row["threshold"]
print(f"Ensemble best threshold: {ens_best_threshold} (F1={ens_best_row['f1']:.4f})")

ensemble_test_decisions = ensemble.predict_batch(
    test_df["context_text"].tolist(), test_df["answer"].tolist()
)
ens_test_preds = [d.confidence >= ens_best_threshold for d in ensemble_test_decisions]
ensemble_result = evaluator.evaluate(ens_test_preds, test_df["label"].tolist())

ensemble.save("../models/ensemble_filter")

all_comparison = evaluator.compare(
    {
        "No Filter": baseline_result,
        "Learned Filter": learned_result,
        "NLI Zero-Shot": nli_result,
        "Ensemble": ensemble_result,
    },
    save_path="../results/filter_comparison_all.json",
)
pd.DataFrame(all_comparison).set_index("strategy").round(4)

## 12. Results Summary

In [ ]:
print("=== FINAL RESULTS (all 6 required metrics) ===")
print(f"Trained on: {len(train_df)} samples, Evaluated on: {len(test_df)} samples")

strategies = [
    ("No Filter", baseline_result, "-"),
    ("Learned (small)", learned_result, best_threshold),
    ("NLI Zero-Shot", nli_result, nli_best_threshold),
    ("Ensemble", ensemble_result, ens_best_threshold),
]

header = (
    f"{'Strategy':<18} {'t':>5} {'P':>6} {'R':>6} {'F1':>6} "
    f"{'Acc':>6} {'RejR':>6} {'RejRate':>8}"
)
print(f"\n{header}")
print("-" * len(header))
for name, r, t in strategies:
    t_str = f"{t}" if isinstance(t, str) else f"{t:.2f}"
    print(
        f"{name:<18} {t_str:>5} {r.precision:>6.3f} {r.recall:>6.3f} "
        f"{r.f1:>6.3f} {r.accuracy:>6.3f} "
        f"{r.rejection_recall:>6.3f} {r.rejection_rate:>8.3f}"
    )

print(f"\nRAG outputs: {n_accept}/{len(rag_df)} accepted ({100*n_accept/len(rag_df):.1f}%)")